# Análisis del Portafolio de Productos por Usuario

Este notebook analiza la tabla `hey_productos.csv` para identificar saldos disponibles, utilización de crédito y compromisos de deuda.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

# Carga de datos según DICCIONARIO_DATOS
BASE_TXN  = Path("Datathon_Hey_2026_dataset_transacciones 1/dataset_transacciones")

df_productos = pd.read_csv(BASE_TXN / "hey_productos.csv",
                           dtype={"producto_id": str, "user_id": str},
                           parse_dates=["fecha_apertura", "fecha_ultimo_movimiento"])

df_clientes = pd.read_csv(BASE_TXN / "hey_clientes.csv", dtype={"user_id": str})

# Filtrar productos activos
df_prod = df_productos[df_productos["estatus"] == "activo"].copy()
print(f"Total de productos activos: {len(df_prod):,}")


## 1. Distribución de tipo_producto

In [ ]:
tipo_prod_counts = df_prod['tipo_producto'].value_counts()
tipo_prod_pct = df_prod['tipo_producto'].value_counts(normalize=True) * 100
dist_df = pd.DataFrame({'Conteo': tipo_prod_counts, 'Porcentaje (%)': tipo_prod_pct})
display(dist_df)

plt.figure(figsize=(10, 6))
ax = sns.barplot(x=tipo_prod_pct.values, y=tipo_prod_pct.index, hue=tipo_prod_pct.index, legend=False)
plt.title('Distribución de Tipos de Producto (%)')
plt.xlabel('Porcentaje del Total (%)')
for i, v in enumerate(tipo_prod_pct.values):
    ax.text(v + 0.5, i, f'{v:.1f}% ({tipo_prod_counts.values[i]:,})', va='center')
plt.tight_layout()
plt.show()


## 2. % de usuarios que tienen inversion_hey activa (Caso UC1)

In [ ]:
total_users = df_prod['user_id'].nunique()
users_with_inversion = df_prod[df_prod['tipo_producto'] == 'inversion_hey']['user_id'].nunique()
pct_inversion = (users_with_inversion / total_users) * 100

print(f"% de usuarios con inversion_hey activa: {pct_inversion:.2f}% ({users_with_inversion:,} de {total_users:,})")

# Cuantificar: ¿qué % de usuarios tiene saldo alternativo disponible en inversion_hey?
inversiones = df_prod[(df_prod["tipo_producto"]=="inversion_hey") & (df_prod["saldo_actual"]>0)]
users_inversion_con_saldo = inversiones['user_id'].nunique()
pct_inversion_con_saldo = (users_inversion_con_saldo / total_users) * 100
print(f"Usuarios con inversión_hey y saldo disponible (>0): {users_inversion_con_saldo:,} ({pct_inversion_con_saldo:.2f}% del total de usuarios con productos activos)")


## 3. Distribución de saldo_actual por tipo_producto

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_prod, x='saldo_actual', y='tipo_producto', hue='tipo_producto', legend=False)
plt.title('Distribución de Saldo Actual por Tipo de Producto')
plt.xscale('symlog')
plt.xlabel('Saldo Actual (escala simétrica logarítmica)')
plt.tight_layout()
plt.show()

display(df_prod.groupby('tipo_producto')['saldo_actual'].describe())


## 4. Distribución de utilizacion_pct para tarjetas de crédito

In [ ]:
tcs = df_prod[df_prod['tipo_producto'].str.contains('tarjeta_credito', case=False, na=False)]

plt.figure(figsize=(10, 6))
sns.histplot(data=tcs, x='utilizacion_pct', bins=50, kde=True)
plt.title('Distribución de Utilización (%) para Tarjetas de Crédito')
plt.xlabel('% de Utilización')
plt.tight_layout()
plt.show()

al_limite = tcs[tcs['utilizacion_pct'] >= 90]
total_tcs_users = tcs['user_id'].nunique()
if total_tcs_users > 0:
    pct = (al_limite['user_id'].nunique() / total_tcs_users) * 100
    print(f"Usuarios con utilización >= 90% (al límite): {al_limite['user_id'].nunique():,} de {total_tcs_users:,} ({pct:.2f}%)")
else:
    print("No hay usuarios con este tipo de producto.")


## 5. Distribución de monto_mensualidad para productos de préstamo

In [ ]:
prestamos = df_prod[df_prod['monto_mensualidad'] > 0]

plt.figure(figsize=(10, 6))
sns.histplot(data=prestamos, x='monto_mensualidad', bins=50, kde=True)
plt.title('Distribución de Monto Mensualidad (Compromisos Fijos)')
plt.xlabel('Monto Mensualidad')
plt.tight_layout()
plt.show()

mediana_deuda = prestamos['monto_mensualidad'].median()
print(f"Mediana de compromisos mensuales de deuda: {mediana_deuda:,.2f}")


## 6. Promedio de num_productos_activos por usuario

In [ ]:
productos_por_usuario = df_prod.groupby('user_id').size().reset_index(name='num_productos_activos')
# Uniendo con clientes por si hay clientes sin productos activos
clientes_prod = pd.merge(df_clientes[['user_id']], productos_por_usuario, on='user_id', how='left').fillna(0)

promedio_prod = clientes_prod['num_productos_activos'].mean()
print(f"Promedio de productos activos por usuario: {promedio_prod:.2f}")


## 7. Usuarios con inversion_hey Y cuenta_debito simultáneamente

In [ ]:
users_inversion = set(df_prod[df_prod['tipo_producto'] == 'inversion_hey']['user_id'])
users_debito = set(df_prod[df_prod['tipo_producto'] == 'cuenta_debito']['user_id'])
users_ambos = users_inversion.intersection(users_debito)

print(f"Usuarios con inversion_hey Y cuenta_debito simultáneamente: {len(users_ambos):,} ({(len(users_ambos)/total_users)*100:.2f}% del total con productos)")
